# Egypt & MENA Digital Financial Inclusion Opportunity Analytics

## SQLite database setup
Load the three processed CSVs into separate tables. Run from `notebooks/`; no source CSV is edited.

## 1. Imports, paths, and source schemas
Inspect actual column names and types before creating the database.

In [1]:
from pathlib import Path
import sqlite3
import pandas as pd

processed_dir = Path('../data/processed')
source_paths = {
    'findex': processed_dir / 'financial_inclusion_long_clean.csv',
    'imf_fas': processed_dir / 'imf_fas_long_clean.csv',
    'cbe_context': processed_dir / 'cbe_egypt_context.csv',
}
database_path = processed_dir / 'financial_inclusion.db'
assert all(path.is_file() for path in source_paths.values())
frames = {table: pd.read_csv(path, low_memory=False) for table, path in source_paths.items()}
for table, frame in frames.items():
    print(f'\n{table}: {frame.shape}')
    print(frame.dtypes.to_string())
    print('columns:', frame.columns.tolist())


findex: (2812, 12)
country                  object
country_code             object
year                      int64
adult_population        float64
region                   object
income_group             object
demographic_group        object
demographic_subgroup     object
indicator_code           object
value                   float64
indicator_name           object
country_role             object
columns: ['country', 'country_code', 'year', 'adult_population', 'region', 'income_group', 'demographic_group', 'demographic_subgroup', 'indicator_code', 'value', 'indicator_name', 'country_role']

imf_fas: (488, 9)
country                 object
year                     int64
indicator_name          object
value                  float64
unit                    object
source_series_code      object
source_country_code     object
source_country_name     object
country_role            object
columns: ['country', 'year', 'indicator_name', 'value', 'unit', 'source_series_code', 'source_country

## 2. Source validation
Check observation keys, measures, and the CBE Egypt-only scope. IMF values retain their original units.

In [2]:
keys = {
    'findex': ['country', 'year', 'demographic_group', 'demographic_subgroup', 'indicator_code'],
    'imf_fas': ['country', 'year', 'indicator_name'],
    'cbe_context': ['country', 'year', 'metric_code'],
}
expected_counts = {'findex': 2812, 'imf_fas': 488, 'cbe_context': 34}
for table, frame in frames.items():
    assert len(frame) == expected_counts[table]
    assert frame[keys[table]].notna().all().all()
    assert not frame.duplicated(keys[table]).any()
    assert pd.api.types.is_integer_dtype(frame.year)
    assert pd.api.types.is_numeric_dtype(frame.value)
    assert frame.value.notna().all()
assert frames['findex'].value.between(0, 1).all()
assert set(frames['cbe_context'].country) == {'Egypt'}
print('Source validation passed.')

Source validation passed.


## 3. Create SQLite tables
The three sources remain in independent tables. Rerunning refreshes these tables from the processed CSVs; indexes support the observation keys and common filters.

In [3]:
with sqlite3.connect(database_path) as connection:
    with connection:
        for table, frame in frames.items():
            frame.to_sql(table, connection, if_exists='replace', index=False)
        connection.execute('CREATE UNIQUE INDEX ux_findex_observation ON findex (country, year, demographic_group, demographic_subgroup, indicator_code)')
        connection.execute('CREATE UNIQUE INDEX ux_imf_observation ON imf_fas (country, year, indicator_name)')
        connection.execute('CREATE UNIQUE INDEX ux_cbe_observation ON cbe_context (country, year, metric_code)')
        connection.execute('CREATE INDEX ix_findex_indicator_year ON findex (indicator_code, year)')
        connection.execute('CREATE INDEX ix_imf_year ON imf_fas (year, indicator_name)')
print('Database saved:', database_path)

Database saved: ../data/processed/financial_inclusion.db


## 4. Loaded-table validation and read-back
Compare every table with the processed CSV, including row order, columns, and numeric values.

In [4]:
with sqlite3.connect(database_path) as connection:
    table_names = [row[0] for row in connection.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name")]
    assert table_names == sorted(frames)
    for table, source in frames.items():
        loaded = pd.read_sql_query(f'SELECT * FROM {table}', connection)
        assert loaded.shape == source.shape
        assert loaded.columns.tolist() == source.columns.tolist()
        assert not loaded.duplicated(keys[table]).any()
        pd.testing.assert_frame_equal(loaded, source, check_dtype=False, rtol=1e-12)
        print(f'{table}: {len(loaded):,} rows; {loaded.country.nunique()} countries; {loaded.year.min()}–{loaded.year.max()}')
    assert connection.execute('PRAGMA integrity_check').fetchone()[0] == 'ok'
print('Read-back and database integrity passed.')

findex: 2,812 rows; 7 countries; 2011–2024
imf_fas: 488 rows; 7 countries; 2004–2025
cbe_context: 34 rows; 1 countries; 2016–2025
Read-back and database integrity passed.


## 5. Methodology
CBE financial-inclusion rates are Egypt-specific context and methodologically distinct from Findex account ownership. IMF measures have their own units. The IMF coverage CSV remains documentation rather than a main SQL table.